# Proposed work pipeline (timed)

In [1]:
import os, sys, json, time
from datetime import datetime

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "src", "config.py")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("Project root (folder containing src/) not found")
    ROOT = parent
sys.path.insert(0, ROOT)
OUT = os.path.join(ROOT, "proposed_work")
os.makedirs(OUT, exist_ok=True)

import numpy as np
from src import config as C
from src.gci_model import pd_given_z, target_gaussian_histogram, monte_carlo_var
from src.quantum_encoding import fit_linear_angle, register_adapted_params
from src.vqc_circuit import build_gci_circuit
from src.training import train_gaussian_loader
from src.device import NoisyDevice
from src.noise_model import ControlDrift
from src.metrics import hellinger_fidelity
from src.postprocessing import credit_statistics, classical_joint_probs

REGISTERS = (2, 3)
P0, RHO, LGD, CONF = C.P0, C.RHO, C.LGD, C.CONFIDENCE

In [2]:
from src.calibration import hierarchical_calibration
from src.mitigation import assignment_matrix, apply_readout_mitigation, run_zne, ReadoutMitigatedDevice
RESULTS = {"pipeline": "proposed_work", "registers": {}}
USE_ZNE = True

In [3]:
TIMINGS = []


def start(stage):
    t = {"stage": stage, "start": datetime.now(), "t0": time.perf_counter()}
    print(f"[START] {stage:45s} {t['start']:%H:%M:%S}")
    return t


def stop(t, circuit_runs=0, shots=0):
    seconds = time.perf_counter() - t["t0"]
    end = datetime.now()
    TIMINGS.append({"stage": t["stage"],
                    "start": t["start"].strftime("%H:%M:%S.%f")[:-3],
                    "end": end.strftime("%H:%M:%S.%f")[:-3],
                    "seconds": round(seconds, 3),
                    "circuit_runs": int(circuit_runs), "shots": int(shots)})
    extra = f"   circuit runs = {circuit_runs}" if circuit_runs else ""
    print(f"[STOP ] {t['stage']:45s} {end:%H:%M:%S}   time = {seconds:.2f} s{extra}")


PIPELINE_START = datetime.now()
PIPELINE_T0 = time.perf_counter()
print("Pipeline started at", PIPELINE_START.strftime("%Y-%m-%d %H:%M:%S"))

Pipeline started at 2026-10-11 11:05:53


In [4]:
t = start("Stage 1 - Classical GCI model")
targets = {n: target_gaussian_histogram(n, C.Z_MAX)[1] for n in REGISTERS}
mc_losses, mc_pdf, mc_cdf, mc_var = monte_carlo_var(P0, RHO, LGD, n_trials=2_000_000, confidence=CONF)
print(f"PD(z=0) = {pd_given_z(0.0, P0, RHO):.4f}   Monte Carlo P(loss=0) = {mc_cdf[0]:.4f}   VaR95 = ${mc_var:.0f}")
stop(t)

[START] Stage 1 - Classical GCI model                 11:05:53


PD(z=0) = 0.2471   Monte Carlo P(loss=0) = 0.7497   VaR95 = $1000
[STOP ] Stage 1 - Classical GCI model                 11:05:53   time = 0.68 s


In [5]:
t = start("Stage 2 - Quantum encoding")
alpha, beta, max_err, mean_err = fit_linear_angle(P0, RHO, C.Z_MAX)
enc = {n: register_adapted_params(n, alpha, beta, C.Z_MAX) for n in REGISTERS}
print(f"alpha = {np.rad2deg(alpha):.2f} deg   beta = {np.rad2deg(beta):.2f} deg   max fit error = {max_err:.4f}")
for n in REGISTERS:
    print(f"  {n}q: alpha~ = {np.rad2deg(enc[n][0]):.2f} deg   beta~ = {np.rad2deg(enc[n][1]):.2f} deg")
stop(t)

[START] Stage 2 - Quantum encoding                    11:05:53
alpha = -3.47 deg   beta = 30.03 deg   max fit error = 0.0065
  2q: alpha~ = -6.93 deg   beta~ = 40.42 deg
  3q: alpha~ = -2.97 deg   beta~ = 40.42 deg
[STOP ] Stage 2 - Quantum encoding                    11:05:53   time = 0.00 s


In [6]:
t = start("Stage 3 - Circuit design")
circuits = {}
for n in REGISTERS:
    a_t, b_t = enc[n]
    circuits[n], _ = build_gci_circuit(n, a_t, b_t)
    print(f"  {n}q: qubits = {circuits[n].num_qubits}   depth = {circuits[n].depth()}   gates = {dict(circuits[n].count_ops())}")
stop(t)

[START] Stage 3 - Circuit design                      11:05:53
  2q: qubits = 3   depth = 6   gates = {'ry': 3, 'measure': 3, 'cry': 2, 'cx': 1, 'barrier': 1}
  3q: qubits = 4   depth = 8   gates = {'ry': 4, 'measure': 4, 'cry': 3, 'cx': 2, 'barrier': 1}
[STOP ] Stage 3 - Circuit design                      11:05:53   time = 0.00 s


In [7]:
t = start("Stage 4 - Classical training (parameter shift + Adam)")
x_nom = {}
for n in REGISTERS:
    theta, _, final_p, final_loss = train_gaussian_loader(n, targets[n], n_iterations=150, lr=0.15, seed=0)
    a_t, b_t = enc[n]
    x_nom[n] = np.array(list(theta) + [b_t, a_t])
    print(f"  {n}q: theta* = {np.round(np.rad2deg(theta), 1)} deg   loss = {final_loss:.2e}")
stop(t)

[START] Stage 4 - Classical training (parameter shift + Adam) 11:05:53


  2q: theta* = [ 90.  195.4] deg   loss = 2.28e-08


  3q: theta* = [ 90.  55. 131.] deg   loss = 4.88e-02
[STOP ] Stage 4 - Classical training (parameter shift + Adam) 11:05:54   time = 0.73 s


In [8]:
t = start("Stage 5 - Transpilation (SABRE) + noisy chip")
dev, ldev, ref, lref = {}, {}, {}, {}
for n in REGISTERS:
    dev[n] = NoisyDevice(n, "gci")
    ldev[n] = NoisyDevice(n, "loader", initial_layout=dev[n].phys[:n])
    ref[n] = dev[n].ideal_probs(x_nom[n])
    lref[n] = ldev[n].ideal_probs(x_nom[n][:n])
    ops = dict(dev[n].tqc.count_ops())
    print(f"  {n}q: transpiled depth = {dev[n].tqc.depth()}   CZ gates = {ops.get('cz', 0)}")
stop(t)

[START] Stage 5 - Transpilation (SABRE) + noisy chip  11:05:54
  2q: transpiled depth = 40   CZ gates = 8
  3q: transpiled depth = 67   CZ gates = 14
[STOP ] Stage 5 - Transpilation (SABRE) + noisy chip  11:05:54   time = 0.05 s


In [9]:
t = start("Stage 6 - Calibration: automated 6B + readout mitigation")
x_cal, A, runs6 = {}, {}, 0
for n in REGISTERS:
    A[n] = assignment_matrix(dev[n])
    M = ReadoutMitigatedDevice(dev[n], A[n])
    LM = ReadoutMitigatedDevice(ldev[n], assignment_matrix(ldev[n]))
    h = hierarchical_calibration(M, ref[n], x_nom[n], LM, lref[n], C.SHOTS_CALIBRATION)
    x_cal[n] = np.asarray(h.x_best)
    rem_runs = 2 ** (n + 1) + 2 ** n
    runs = h.n_evals + rem_runs
    runs6 += runs
    RESULTS["registers"][f"{n}q"] = {"calibration_runs": h.n_evals, "rem_calibration_runs": rem_runs}
    print(f"  {n}q: automated calibration used {h.n_evals} circuit runs (+{rem_runs} readout-calibration runs)")
stop(t, runs6, runs6 * C.SHOTS_CALIBRATION)

[START] Stage 6 - Calibration: automated 6B + readout mitigation 11:05:54


  2q: automated calibration used 204 circuit runs (+12 readout-calibration runs)


  3q: automated calibration used 204 circuit runs (+24 readout-calibration runs)
[STOP ] Stage 6 - Calibration: automated 6B + readout mitigation 11:06:00   time = 6.22 s   circuit runs = 444


In [10]:
t = start("Stage 7 - Execution + readout mitigation")
p_exec, f_exec = {}, {}
for n in REGISTERS:
    reps = [apply_readout_mitigation(dev[n].run(x_cal[n], C.SHOTS_EXECUTION, seed=500 + 1000 * r), A[n])
            for r in range(C.N_REPEATS)]
    p_exec[n] = np.mean(reps, axis=0)
    f_exec[n] = [hellinger_fidelity(p, ref[n]) for p in reps]
    print(f"  {n}q: {C.N_REPEATS} repeats x {C.SHOTS_EXECUTION} shots   F_H = {np.mean(f_exec[n]):.4f} +/- {np.std(f_exec[n]):.4f}")
runs7 = len(REGISTERS) * C.N_REPEATS
stop(t, runs7, runs7 * C.SHOTS_EXECUTION)

[START] Stage 7 - Execution + readout mitigation      11:06:01


  2q: 20 repeats x 20000 shots   F_H = 0.9820 +/- 0.0014


  3q: 20 repeats x 20000 shots   F_H = 0.9939 +/- 0.0006
[STOP ] Stage 7 - Execution + readout mitigation      11:06:02   time = 1.53 s   circuit runs = 40


In [11]:
if USE_ZNE:
    t = start("Stage 7b - Zero-noise extrapolation (add-on)")
    f_zne = {}
    for n in REGISTERS:
        reps = [run_zne(dev[n], x_cal[n], C.SHOTS_EXECUTION, C.ZNE_SCALE_FACTORS, seed=700 + 1000 * r, readout_A=A[n])[0]
                for r in range(C.N_REPEATS)]
        f_zne[n] = [hellinger_fidelity(p, ref[n]) for p in reps]
        RESULTS["registers"][f"{n}q"]["fidelity_zne"] = float(np.mean(f_zne[n]))
        print(f"  {n}q: F_H with ZNE = {np.mean(f_zne[n]):.4f} +/- {np.std(f_zne[n]):.4f}")
    runs7b = len(REGISTERS) * C.N_REPEATS * len(C.ZNE_SCALE_FACTORS)
    stop(t, runs7b, runs7b * C.SHOTS_EXECUTION)

[START] Stage 7b - Zero-noise extrapolation (add-on)  11:06:02


  2q: F_H with ZNE = 0.9970 +/- 0.0009


  3q: F_H with ZNE = 0.9926 +/- 0.0011
[STOP ] Stage 7b - Zero-noise extrapolation (add-on)  11:06:07   time = 5.30 s   circuit runs = 120


In [12]:
t = start("Stage 8 - Classical post-processing")
stats, classical = {}, {}
for n in REGISTERS:
    stats[n] = credit_statistics(p_exec[n], n, lgd=LGD, confidence=CONF)
    classical[n] = credit_statistics(classical_joint_probs(n, P0, RHO), n, lgd=LGD, confidence=CONF)
    q, c = stats[n], classical[n]
    print(f"  {n}q: PD = {q['p_default']:.4f} (classical {c['p_default']:.4f})   "
          f"E[L] = ${q['expected_loss']:.1f} (classical ${c['expected_loss']:.1f})   "
          f"VaR95 = ${q['var']:.0f} (classical ${c['var']:.0f})")
stop(t)

[START] Stage 8 - Classical post-processing           11:06:07
  2q: PD = 0.2582 (classical 0.2504)   E[L] = $258.2 (classical $250.4)   VaR95 = $1000 (classical $1000)
  3q: PD = 0.2543 (classical 0.2500)   E[L] = $254.3 (classical $250.0)   VaR95 = $1000 (classical $1000)
[STOP ] Stage 8 - Classical post-processing           11:06:07   time = 0.00 s


In [13]:
t = start("Stage 9 - Fidelity gate (ACCEPT / RE-CALIBRATE)")
for n in REGISTERS:
    f_ideal = float(np.mean(f_exec[n]))
    f_class = hellinger_fidelity(p_exec[n], classical_joint_probs(n, P0, RHO))
    decision = "ACCEPT" if f_ideal >= C.FIDELITY_THRESHOLD else "RE-CALIBRATE"
    RESULTS["registers"][f"{n}q"].update({
        "fidelity_vs_ideal": f_ideal, "fidelity_vs_ideal_std": float(np.std(f_exec[n])),
        "fidelity_vs_classical": float(f_class),
        "p_default": stats[n]["p_default"], "expected_loss": stats[n]["expected_loss"],
        "var95": stats[n]["var"], "classical_var95": classical[n]["var"],
        "var_matches_classical": stats[n]["var"] == classical[n]["var"],
        "decision": decision})
    print(f"  {n}q: F_H vs ideal = {f_ideal:.4f}   F_H vs classical = {f_class:.4f}   "
          f"threshold {C.FIDELITY_THRESHOLD}  ->  {decision}")
stop(t)

[START] Stage 9 - Fidelity gate (ACCEPT / RE-CALIBRATE) 11:06:07
  2q: F_H vs ideal = 0.9820   F_H vs classical = 0.9824   threshold 0.97  ->  ACCEPT
  3q: F_H vs ideal = 0.9939   F_H vs classical = 0.8205   threshold 0.97  ->  ACCEPT
[STOP ] Stage 9 - Fidelity gate (ACCEPT / RE-CALIBRATE) 11:06:07   time = 0.00 s


In [14]:
t = start("Drift event - automatic detection + recalibration")
DAY2 = ControlDrift(C.DRIFT_EPS_DAY2, C.DRIFT_DELTA_DEG_DAY2)
runs_d = 0
for n in REGISTERS:
    dev2 = NoisyDevice(n, "gci", drift=DAY2)
    ldev2 = NoisyDevice(n, "loader", drift=DAY2, initial_layout=dev2.phys[:n])
    A2 = assignment_matrix(dev2)
    M2 = ReadoutMitigatedDevice(dev2, A2)
    LM2 = ReadoutMitigatedDevice(ldev2, assignment_matrix(ldev2))
    x = x_cal[n].copy()
    runs = 2 ** (n + 1) + 2 ** n
    log = []
    for r in range(C.MAX_RECALIBRATION_ROUNDS + 1):
        f = hellinger_fidelity(M2.run(x, C.SHOTS_FINAL_CHECK, seed=900 + r), ref[n])
        runs += 1
        decision = "ACCEPT" if f >= C.FIDELITY_THRESHOLD else "RE-CALIBRATE"
        log.append({"round": r, "fidelity": float(f), "decision": decision})
        print(f"  {n}q round {r}: F_H = {f:.4f}  ->  {decision}")
        if decision == "ACCEPT" or r == C.MAX_RECALIBRATION_ROUNDS:
            break
        h = hierarchical_calibration(M2, ref[n], x, LM2, lref[n], C.SHOTS_CALIBRATION, seed=r + 1)
        runs += h.n_evals
        x = np.asarray(h.x_best)
    runs_d += runs
    RESULTS["registers"][f"{n}q"]["drift"] = {"fidelity_before": log[0]["fidelity"],
                                               "fidelity_after": log[-1]["fidelity"],
                                               "recovery_runs": runs, "rounds": len(log) - 1, "log": log}
stop(t, runs_d, runs_d * C.SHOTS_CALIBRATION)

[START] Drift event - automatic detection + recalibration 11:06:07


  2q round 0: F_H = 0.6285  ->  RE-CALIBRATE


  2q round 1: F_H = 0.9606  ->  RE-CALIBRATE


  2q round 2: F_H = 0.9758  ->  ACCEPT


  3q round 0: F_H = 0.6960  ->  RE-CALIBRATE


  3q round 1: F_H = 0.9883  ->  ACCEPT
[STOP ] Drift event - automatic detection + recalibration 11:06:16   time = 9.10 s   circuit runs = 653


In [15]:
TOTAL_SECONDS = time.perf_counter() - PIPELINE_T0
PIPELINE_END = datetime.now()
print("Pipeline started :", PIPELINE_START.strftime("%H:%M:%S"))
print("Pipeline finished:", PIPELINE_END.strftime("%H:%M:%S"))
print(f"TOTAL TIME       : {TOTAL_SECONDS:.2f} s")
print()
print(f"{'stage':48s} {'start':>12s} {'end':>12s} {'seconds':>9s} {'runs':>6s}")
for r in TIMINGS:
    print(f"{r['stage']:48s} {r['start']:>12s} {r['end']:>12s} {r['seconds']:9.2f} {r['circuit_runs']:6d}")

with open(os.path.join(OUT, "timing.json"), "w") as f:
    json.dump({"pipeline": "proposed_work",
               "started": PIPELINE_START.strftime("%Y-%m-%d %H:%M:%S"),
               "finished": PIPELINE_END.strftime("%Y-%m-%d %H:%M:%S"),
               "total_seconds": round(TOTAL_SECONDS, 3), "stages": TIMINGS}, f, indent=2)
with open(os.path.join(OUT, "results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, default=float)
print("Saved ->", os.path.join(OUT, "timing.json"))
print("Saved ->", os.path.join(OUT, "results.json"))

Pipeline started : 11:05:53
Pipeline finished: 11:06:16
TOTAL TIME       : 23.73 s

stage                                                   start          end   seconds   runs
Stage 1 - Classical GCI model                    11:05:53.256 11:05:53.940      0.68      0
Stage 2 - Quantum encoding                       11:05:53.950 11:05:53.952      0.00      0
Stage 3 - Circuit design                         11:05:53.958 11:05:53.960      0.00      0
Stage 4 - Classical training (parameter shift + Adam) 11:05:53.968 11:05:54.700      0.73      0
Stage 5 - Transpilation (SABRE) + noisy chip     11:05:54.711 11:05:54.764      0.05      0
Stage 6 - Calibration: automated 6B + readout mitigation 11:05:54.773 11:06:00.993      6.22    444
Stage 7 - Execution + readout mitigation         11:06:01.003 11:06:02.535      1.53     40
Stage 7b - Zero-noise extrapolation (add-on)     11:06:02.546 11:06:07.846      5.30    120
Stage 8 - Classical post-processing              11:06:07.853 11:06:07.855 